# Data Quality Report

Eye-tracking datasets can contain a variety of structural problems that propagate
silently through preprocessing pipelines and corrupt downstream analyses. This
tutorial demonstrates `Dataset.report_data_quality()`, which addresses
[GitHub Issue #1386](https://github.com/pymovements/pymovements/issues/1386)
by providing a single method that:

1. **Validates** the dataset configuration against the loaded gaze data (seven checks).
2. **Measures** data quality (data loss, fixation precision) at dataset, subject,
   session, and trial level.
3. **Saves** BIDS-conformant derivative report files for archiving and sharing.

### Failure modes covered

| Class | Example | Effect |
|-------|---------|--------|
| Column name absent | `trial_columns=['trialId']` but frame has `'trial_id'` | Silent corrupt event timestamps |
| Wrong dtype | Trial column stored as `Float64` after CSV parsing | Group-by join ambiguity |
| No gaze components | Neither `pixel_columns` nor `position_columns` set | Downstream transform failure |

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np
import polars as pl

from pymovements.dataset.data_quality import (_ALL_CHECKS, DataQualityReport,
                                              GazeDataValidationError,
                                              _compute_measures)
from pymovements.gaze.experiment import Experiment
from pymovements.gaze.gaze import Gaze

## 1. Setting Up a Minimal Gaze Object

For this tutorial we build a `Gaze` object directly from a synthetic
`polars.DataFrame`. This avoids downloading a public dataset while still
exercising the full API.

The helper below constructs a `Gaze` with 200 samples at 1000 Hz, two trials,
and a position column in degrees of visual angle.

In [ ]:
def make_clean_gaze(n: int = 200, sampling_rate: float = 1000.0) -> Gaze:
    """Synthetic gaze with 2 trials, proper timestamps, and position column."""
    rng = np.random.default_rng(42)
    time = list(range(0, n))
    # Two trials of equal length
    trial = [1] * (n // 2) + [2] * (n // 2)
    # Gaze around screen centre with small jitter
    x = (0.0 + rng.normal(0, 0.3, n)).tolist()
    y = (0.0 + rng.normal(0, 0.2, n)).tolist()

    experiment = Experiment(
        screen_width_px=1280,
        screen_height_px=1024,
        screen_width_cm=38.0,
        screen_height_cm=30.0,
        distance_cm=68.0,
        origin='center',
        sampling_rate=sampling_rate,
    )

    gaze = Gaze(
        samples=pl.DataFrame({
            'time': time,
            'trial_id': trial,
            'pos_x': x,
            'pos_y': y,
        }),
        trial_columns='trial_id',
        time_column='time',
        position_columns=['pos_x', 'pos_y'],
        experiment=experiment,
    )
    return gaze


clean_gaze = make_clean_gaze()
print('Columns:', clean_gaze.samples.columns)
print('Shape:  ', clean_gaze.samples.shape)
clean_gaze.samples.head(4)

## 2. Running `report_data_quality()` on a Clean Dataset

We attach the gaze object to a minimal dataset-like namespace and call
`report_data_quality()` directly. In a real workflow you would use
`dataset.load()` first and then call the method on the `Dataset` instance.

In [ ]:
def run_report(
    gaze_list,
    checks=None,
    measures=None,
    levels=None,
    raise_on_error=False,
    output_path=None,
):
    """Thin wrapper that replicates Dataset.report_data_quality() logic."""
    import warnings as _warnings

    checks_to_run = checks or list(_ALL_CHECKS.keys())
    levels_to_run = levels or ['dataset', 'subject', 'session', 'trial']

    report = DataQualityReport()
    captured = []

    with _warnings.catch_warnings(record=True) as caught:
        _warnings.simplefilter('always')
        for check_id in checks_to_run:
            check_fn = _ALL_CHECKS[check_id]
            for idx, gaze in enumerate(gaze_list):
                result = check_fn(gaze, str(idx))
                report.check_results.append(result)
                if raise_on_error and result.severity == 'error':
                    raise GazeDataValidationError(
                        check_id=result.check_id,
                        message=str(result.message),
                        affected_files=result.affected_files,
                    )
        report.measures = _compute_measures(gaze_list, {}, levels_to_run, measures)
        captured = [str(w.message) for w in caught]

    report._warning_log = captured
    if output_path is not None:
        report.save_bids_report(Path(output_path))
    return report


report_clean = run_report([clean_gaze])
print('passed:', report_clean.passed)
print()
print(report_clean.summary())

## 3. Inspecting the ValidationReport

Each check produces a `CheckResult` with a `severity` of `'pass'`, `'warning'`,
or `'error'` and a human-readable `message`.

In [ ]:
for r in report_clean.check_results:
    icon = {'pass': '✓', 'warning': '⚠', 'error': '✗'}.get(r.severity, '?')
    print(f'{icon}  [{r.severity:<8}]  {r.check_id:<35}  {r.message[:80]}')

### Quality measures

`report.measures` is a dict keyed by aggregation level.
Each value is a `polars.DataFrame` with data loss and precision columns.

In [ ]:
for level, df in report_clean.measures.items():
    print(f'\n--- {level} level ---')
    print(df)

## 4. Demonstrating the Three Failure Classes

### Failure Class 1: trial column absent from the schema

The most dangerous failure: `trial_columns` names a column that does not exist
in the underlying frame. Event detection silently crosses trial boundaries.

In [ ]:
# Correct column name is 'trial_id'; we accidentally declare 'trialId'
broken_gaze_1 = Gaze(
    samples=pl.DataFrame({
        'time': list(range(10)),
        'trial_id': [1] * 5 + [2] * 5,
        'pos_x': [0.0] * 10,
        'pos_y': [0.0] * 10,
    }),
    time_column='time',
    trial_columns='trialId',       # <-- case mismatch!
    position_columns=['pos_x', 'pos_y'],
    experiment=Experiment(1280, 1024, 38, 30, 68, 'center', sampling_rate=1000.0),
)

report_broken_1 = run_report([broken_gaze_1], checks=['trial_columns_exist'])
r = report_broken_1.check_results[0]
print(f'severity : {r.severity}')
print(f'message  : {r.message}')
print()
print('Corrective action: change trial_columns to the actual column name.')
print("  Correct: trial_columns='trial_id'")

### Failure Class 2: trial column has float dtype

CSV parsers sometimes infer numeric columns as `Float64`. Trial identifiers
stored as floats cause group-by ambiguity in downstream reading measure
aggregations.

In [ ]:
broken_gaze_2 = Gaze(
    samples=pl.DataFrame({
        'time': list(range(10)),
        'trial_id': pl.Series([1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 2.0, 2.0, 2.0],
                              dtype=pl.Float64),   # <-- float dtype!
        'pos_x': [0.0] * 10,
        'pos_y': [0.0] * 10,
    }),
    time_column='time',
    trial_columns='trial_id',
    position_columns=['pos_x', 'pos_y'],
    experiment=Experiment(1280, 1024, 38, 30, 68, 'center', sampling_rate=1000.0),
)

report_broken_2 = run_report([broken_gaze_2], checks=['trial_columns_dtype'])
r = report_broken_2.check_results[0]
print(f'severity : {r.severity}')
print(f'message  : {r.message}')
print()
print('Corrective action: cast trial column to integer.')
print("  broken_gaze_2.samples = broken_gaze_2.samples.with_columns(")
print("      pl.col('trial_id').cast(pl.Int64)")
print("  )")

### Failure Class 3: no gaze components defined

If neither `pixel_columns`, `position_columns`, nor `velocity_columns` is set,
all downstream transformations (`pix2deg`, `pos2vel`) will fail.

In [ ]:
broken_gaze_3 = Gaze(
    samples=pl.DataFrame({
        'time': list(range(5)),
        'trial_id': [1, 1, 2, 2, 2],
        # No coordinate columns at all
    }),
    time_column='time',
    trial_columns='trial_id',
    experiment=Experiment(1280, 1024, 38, 30, 68, 'center', sampling_rate=1000.0),
)

report_broken_3 = run_report([broken_gaze_3], checks=['gaze_components_defined'])
r = report_broken_3.check_results[0]
print(f'severity : {r.severity}')
print(f'message  : {r.message}')
print()
print('Corrective action: specify coordinate columns during Gaze initialisation.')
print("  position_columns=['pos_x', 'pos_y']")

## 5. `raise_on_error` Mode

By default errors are collected into the report without stopping execution.
Set `raise_on_error=True` to implement a fail-fast pipeline.

In [ ]:
try:
    run_report(
        [broken_gaze_1],
        checks=['trial_columns_exist'],
        raise_on_error=True,
    )
except GazeDataValidationError as exc:
    print('GazeDataValidationError raised!')
    print(f'  check_id       : {exc.check_id}')
    print(f'  affected_files : {exc.affected_files}')
    print(f'  message        : {exc}')

## 6. Saving a BIDS-Conformant Report

Pass `output_path` to write derivative files following the
[BIDS derivatives specification](https://bids-specification.readthedocs.io/en/stable/derivatives/introduction.html).

Files are written under `output_path/derivatives/pymovements/`.

In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir:
    report_clean_bids = run_report([clean_gaze], output_path=tmp_dir)

    deriv = Path(tmp_dir) / 'derivatives' / 'pymovements'
    print('Files written:')
    for f in sorted(deriv.iterdir()):
        print(f'  {f.name}')

    print()
    print('--- dataset_description.json ---')
    desc = json.loads((deriv / 'dataset_description.json').read_text())
    print(json.dumps(desc, indent=2))

    print()
    print('--- data_quality_checks.tsv (first 5 rows) ---')
    checks_df = pl.read_csv(deriv / 'data_quality_checks.tsv', separator='\t')
    print(checks_df.head(5))

    if (deriv / 'data_quality_measures_dataset.tsv').exists():
        print()
        print('--- data_quality_measures_dataset.tsv ---')
        measures_df = pl.read_csv(
            deriv / 'data_quality_measures_dataset.tsv', separator='\t',
        )
        print(measures_df)

## 7. Selective Checks and Measures

Use `checks=` and `measures=` to request only a subset, which is useful for
quick pipelines or when only specific aspects of data quality are relevant.

In [ ]:
# Run only the two error-severity checks
report_errors_only = run_report(
    [clean_gaze],
    checks=['trial_columns_exist', 'time_column_exists', 'gaze_components_defined'],
    measures=['data_loss'],
    levels=['dataset'],
)
print(f'Number of check results : {len(report_errors_only.check_results)}')
print(f'Measure levels computed : {list(report_errors_only.measures.keys())}')
print()
print(report_errors_only.summary())

## Summary

| Feature | API |
|---------|-----|
| Run all 7 checks + 4 measures | `dataset.report_data_quality()` |
| Select specific checks | `checks=['trial_columns_exist', ...]` |
| Select specific measures | `measures=['data_loss', 'std_rms']` |
| Choose aggregation levels | `levels=['dataset', 'trial']` |
| Fail-fast on first error | `raise_on_error=True` |
| Write BIDS derivative files | `output_path='/path/to/bids/root'` |

### Validation checks (7 total)

| Identifier | Severity | What it detects |
|-----------|----------|----------------|
| `trial_columns_exist` | **error** | Declared trial column absent from schema |
| `trial_columns_dtype` | warning | Trial column has float dtype |
| `time_column_exists` | **error** | No numeric `time` column |
| `gaze_components_defined` | **error** | No coordinate column at all |
| `trial_continuity` | warning | Non-monotone timestamps or large gaps |
| `sampling_rate_consistency` | warning | Empirical rate deviates >5% from declared |
| `gaze_range` | warning | <95% of samples within screen bounds |

### Quality measures (4 total)

| Measure | Description |
|---------|-------------|
| `data_loss` | Fraction of null/invalid samples |
| `std_rms` | RMS standard deviation of position (fixation precision) |
| `rms_s2s` | RMS of sample-to-sample displacements (positional noise) |
| `bcea` | Bivariate contour ellipse area at 68.27% confidence |

See the [BIDS derivatives specification](https://bids-specification.readthedocs.io/en/stable/derivatives/introduction.html)
for details on the output file format.